# 07｜因子動物園：當學術論文太多，你該怎麼辦？

**學術來源：**
- Harvey, Liu & Zhu (2016), *… and the Cross-Section of Expected Returns*, Review of Financial Studies
- Cochrane (2011), *Discount Rates*（AFA 主席演講）

截至 2023 年，學術論文已發表超過 **400 個** 聲稱能預測股票報酬的「因子」。
這個現象被 Cochrane（2011）戲稱為 **因子動物園（Factor Zoo）**。

問題核心：**這些因子是真實的嗎？還是純粹的資料探勘（Data Mining）？**

## 「這個因子去年回測報酬 300%」

你有沒有看過這類標題？

學術界也一樣。每年有幾百篇論文宣稱發現了新的「超額報酬因子」：

- 小型股效應（Fama & French, 1992）——有道理，經典。
- 低波動異象（Black, 1972）——反直覺，但數據說確實存在。
- 公司名字裡有「.com」的股票（2000 年代真實研究）——嗯……

問題是，學術期刊上被發表的因子已經超過 **400 個**。

難道市場真的有 400 個賺錢的規律？

還是大多數只是「**資料探勘**」的產物——在夠大的數據集裡，只要測夠多的組合，
總會找到幾個「剛好有效」的假規律？

2016 年，Harvey、Liu & Zhu 寫了一篇讓整個量化社群反省的論文。
這一章，我們來理解為什麼「在歷史數據上有效」和「未來真的賺錢」之間，
距離可能比你想的大很多。

## 🎯 學習目標
1. 理解「多重檢定問題」如何讓假因子看起來顯著
2. 掌握 Harvey et al. (2016) 建議的 t > 3.0 篩選標準
3. 建立評估「投資策略是否可信」的完整檢核清單

## 1｜多重檢定問題（Multiple Testing Problem）

**傳統統計顯著性：** t > 1.96（α = 5%）

如果你同時測試 100 個策略：
- 即使所有策略都沒有真實效果（全是雜訊）
- 平均仍有 **5 個** 會偶然達到 t > 1.96

Harvey et al.（2016）建議：在考慮多重檢定後，新因子的閾值應提高到 **t > 3.0**。

他們的資料庫中，歷史上發表的因子有 **85%** 的 t 值在 1.96–3.0 之間，
這些因子可能只是統計噪音，而非真實的市場異象。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = [
    'Microsoft YaHei', 'SimHei', 'Heiti TC',
    'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif'
]
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from scipy import stats
import warnings, pathlib
warnings.filterwarnings('ignore')

DATA_DIR = pathlib.Path('data')
rng = np.random.default_rng(42)

# 模擬：在完全隨機數據中，多重檢定產生多少假陽性
N_FACTORS  = 1000  # 測試因子數
N_MONTHS   = 120   # 每個因子測試 120 個月

t_stats = []
for _ in range(N_FACTORS):
    # 純隨機因子，真實超額報酬 = 0
    returns = rng.normal(0, 1, N_MONTHS)
    t_stat  = returns.mean() / (returns.std() / np.sqrt(N_MONTHS))
    t_stats.append(abs(t_stat))

t_stats = np.array(t_stats)

fp_196 = (t_stats > 1.96).sum()
fp_300 = (t_stats > 3.00).sum()
print(f'在 {N_FACTORS} 個純隨機因子中：')
print(f'  t > 1.96（傳統 5% 顯著）：{fp_196} 個 ({fp_196/N_FACTORS:.1%})')
print(f'  t > 3.00（Harvey建議）   ：{fp_300} 個 ({fp_300/N_FACTORS:.1%})')

## 2｜視覺化：t 統計量的分布與假陽性區域

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.hist(t_stats, bins=50, color='#2196F3', alpha=0.7, edgecolor='white', label='隨機因子 t 統計量')

# 理論卡方分配（null hypothesis 下的 |t| 分配近似）
x = np.linspace(0, t_stats.max() + 0.5, 200)
# 使用摺疊常態（|t| 的近似）
folded = stats.norm.pdf(x) * 2 * N_FACTORS * (t_stats.max() / 10)

ax.axvline(1.96, color='#FF9800', linewidth=2, linestyle='--', label=f't=1.96（傳統 5%）: {fp_196} 個假陽性')
ax.axvline(3.00, color='#F44336', linewidth=2, linestyle='--', label=f't=3.00（Harvey 建議）: {fp_300} 個假陽性')

# 標注假陽性區域
ymax = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 50
hist_vals, hist_edges = np.histogram(t_stats, bins=50)
ymax = hist_vals.max()

ax.fill_betweenx([0, ymax * 1.05], 1.96, 3.00, alpha=0.1, color='#FF9800', label='傳統顯著但 Harvey 不認可的區域')
ax.fill_betweenx([0, ymax * 1.05], 3.00, t_stats.max() + 0.5, alpha=0.1, color='#F44336', label='Harvey 認可的顯著區域')

ax.set_xlabel('|t 統計量|', fontsize=12)
ax.set_ylabel('因子數量', fontsize=12)
ax.set_title(f'多重檢定模擬：{N_FACTORS} 個純隨機因子的 t 統計量分布\n（真實超額報酬 = 0，但傳統標準仍有 {fp_196/N_FACTORS:.1%} 假陽性）',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('data/factor_zoo_tstat.png', dpi=150, bbox_inches='tight')
plt.show()

## 3｜真實因子的 t 統計量 vs Harvey 閾值

In [ ]:
# 載入所有已快取的因子
factor_files = {
    'FF3 (Mkt-RF, SMB, HML)': DATA_DIR / 'ff3_factors.csv',
    'FF5 (RMW, CMA)':         DATA_DIR / 'ff5_factors.csv',
    'Momentum':                DATA_DIR / 'ff_momentum.csv',
}

all_factors = {}
for desc, fpath in factor_files.items():
    if fpath.exists():
        tmp = pd.read_csv(fpath, index_col=0, parse_dates=True)
        for col in tmp.columns:
            if col != 'RF':
                all_factors[col] = tmp[col]

factor_df = pd.DataFrame(all_factors).dropna()

t_real = {}
for col in factor_df.columns:
    s = factor_df[col]
    t_val = s.mean() / (s.std() / np.sqrt(len(s)))
    t_real[col] = abs(t_val)

t_series = pd.Series(t_real).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors_bar = ['#4CAF50' if v > 3.0 else '#FF9800' if v > 1.96 else '#F44336'
              for v in t_series]
bars = ax.barh(t_series.index, t_series.values, color=colors_bar, alpha=0.85)
ax.axvline(1.96, color='#FF9800', linewidth=1.5, linestyle='--', label='t=1.96（傳統）')
ax.axvline(3.00, color='#4CAF50', linewidth=1.5, linestyle='--', label='t=3.00（Harvey）')
for bar, val in zip(bars, t_series.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10)
ax.set_xlabel('|t 統計量|（全樣本期間）', fontsize=11)
ax.set_title('已知學術因子的 t 統計量\n（綠色 = Harvey 認可，橘色 = 傳統顯著但可疑）',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('data/factor_tstats.png', dpi=150, bbox_inches='tight')
plt.show()

## 4｜因子報酬的時間不穩定性：滾動夏普比率

In [ ]:
window = 120  # 10 年
fig, axes = plt.subplots(len(factor_df.columns), 1,
                          figsize=(12, 3 * len(factor_df.columns)),
                          sharex=True)
if len(factor_df.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, factor_df.columns):
    s = factor_df[col]
    roll_sh = (s.rolling(window).mean() * 12) / (s.rolling(window).std() * np.sqrt(12))
    ax.plot(roll_sh.index, roll_sh, linewidth=1.2, color='#2196F3')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(roll_sh.mean(), color='orange', linewidth=1.0,
               linestyle='--', label=f'平均={roll_sh.mean():.2f}')
    ax.fill_between(roll_sh.index, 0, roll_sh,
                    where=(roll_sh >= 0), alpha=0.15, color='#4CAF50')
    ax.fill_between(roll_sh.index, 0, roll_sh,
                    where=(roll_sh < 0),  alpha=0.15, color='#F44336')
    pct_neg = (roll_sh.dropna() < 0).mean()
    ax.set_title(f'{col} — 滾動10年夏普比率（{pct_neg:.0%}時段為負）',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle('所有因子的時間不穩定性（滾動10年夏普）', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/factor_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

## 5｜這跟你有什麼關係？

### 評估「投資策略」的防騙檢核清單

當有人告訴你某個因子/策略「學術研究證實有效」，請用以下問題過濾：

#### 統計層面
- [ ] **t 統計量 > 3.0 嗎？** 只達到 1.96 的策略有很高機率是資料探勘
- [ ] **樣本外（out-of-sample）驗證了嗎？** 在其他國家、其他時期仍有效嗎？
- [ ] **有多少個人測試了幾個策略？** 10 個人各測 100 個，一定有人找到「顯著」策略

#### 經濟層面
- [ ] **有合理的經濟解釋嗎？** 純粹的數字遊戲沒有持久性
- [ ] **考慮交易成本了嗎？** 許多高換手率因子在扣費後消失
- [ ] **公布後報酬是否縮水？** McLean & Pontiff（2016）：因子被發表後，平均報酬下降 32%

#### 時間層面
- [ ] **過去10年每個10年滾動期都有效嗎？** 還是只靠某個特殊時期撐起來的？
- [ ] **最大回撤期有多長？** 能撐過5年以上的連續虧損嗎？

### 倖存偏差（Survivorship Bias）

你在網路上看到的「有效策略」，都是存活下來的——那些失敗的策略沉默地消失了。

這不代表所有策略無效，而是：**做好統計衛生，要求更高的證明標準**。

### 底線建議

| 策略類型 | 建議 |
|----------|------|
| 有60年數據支撐、t>3、多國驗證 | Mkt、SMB、HML、RMW、Mom — 可考慮 |
| 學術論文支撐但 t 在 1.96–3.0 | 存疑，等待更多驗證 |
| 回測表現很好但沒有經濟解釋 | 大概率是資料探勘，避開 |
| 只有近5年數據 | 樣本太小，無法判斷 |

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| 因子動物園 | 已發表 400+ 因子，大多數是統計假象 |
| t > 1.96 的問題 | 1000個隨機策略仍有 ~50個 通過傳統顯著性 |
| Harvey 門檻 | t > 3.0 才有較高可信度 |
| 發表後衰退 | 因子被公開後平均報酬下降 32% |
| 個人應用 | 要求：樣本外驗證 + 經濟直覺 + t > 3 |

> **下一章：** Fama-French 三因子——規模與價值的學術基礎